In [1]:
import requests
import supervision as sv
from PIL import Image
from rfdetr import RFDETRMedium
from rfdetr.util.coco_classes import COCO_CLASSES

model = RFDETRMedium()

image = Image.open("C:/Users/thoma/dev_projet/ecptr/test RF/cat-2068462_1280.jpg")

image.show()

detections = model.predict(image, threshold=0.5)

print(detections)

labels = [
    f"{COCO_CLASSES[class_id]}"
    for class_id
    in detections.class_id
]

annotated_image = sv.BoxAnnotator().annotate(image, detections)

annotated_image.show()
annotated_image = sv.LabelAnnotator().annotate(annotated_image, detections, labels)
annotated_image.show()


Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
Loading pretrain weights


Model is not optimized for inference. Latency may be higher than expected. You can optimize the model for inference by calling model.optimize_for_inference().


Detections(xyxy=array([[ 398.817   ,   90.57137 ,  962.4303  ,  810.0343  ],
       [ 943.45325 ,  108.624825, 1280.6536  ,  590.1661  ],
       [ 268.70804 ,  553.48096 , 1280.0505  ,  843.77466 ]],
      dtype=float32), mask=None, confidence=array([0.97291654, 0.9293702 , 0.8395456 ], dtype=float32), class_id=array([17, 62, 67]), tracker_id=None, data={}, metadata={})


In [ ]:
from rfdetr import RFDETRMedium

model = RFDETRMedium()

model.train(
    dataset_dir="result.json",
    epochs=100,
    batch_size=4,
    grad_accum_steps=4,
    lr=1e-4,
    output_dir="output",
)

In [9]:
import json
import random
import shutil
from pathlib import Path

def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2))

def split_coco_80_20(
    coco_json_path: str,
    images_dir: str,
    output_root: str,
    use_symlinks: bool = False
):
    coco_json_path = Path(coco_json_path)
    images_dir = Path(images_dir)
    output_root = Path(output_root)

    coco = json.loads(coco_json_path.read_text())

    images = coco["images"]
    annotations = coco["annotations"]
    categories = coco.get("categories", [])
    info = coco.get("info", {})
    licenses = coco.get("licenses", [])

    # Shuffle images deterministically
    rng = random.Random()
    images_shuffled = images.copy()
    rng.shuffle(images_shuffled)

    n = len(images_shuffled)
    n_train = int(0.8 * n)
    train_images = images_shuffled[:n_train]
    val_images = images_shuffled[n_train:]

    # Helper to build split
    def build_split(split_name: str, split_images: list):
        split_dir = output_root / split_name
        split_img_dir = split_dir / "images"
        split_img_dir.mkdir(parents=True, exist_ok=True)

        # Old image_id -> new image_id (1..)
        old_to_new_img_id = {}
        new_images = []
        for new_id, img in enumerate(split_images, start=1):
            old_to_new_img_id[img["id"]] = new_id
            new_img = dict(img)
            new_img["id"] = new_id
            new_images.append(new_img)

            # Copy/symlink image file
            src = images_dir / img["file_name"]
            dst = split_img_dir / img["file_name"]
            if not src.exists():
                raise FileNotFoundError(f"Image manquante: {src}")

            if dst.exists():
                continue

            if use_symlinks:
                # Symlink (rapide, mais parfois moins portable sous Windows)
                dst.symlink_to(src.resolve())
            else:
                shutil.copy2(src, dst)

        # Filter + remap annotations
        new_annotations = []
        ann_id = 1
        split_old_ids = set(old_to_new_img_id.keys())

        for ann in annotations:
            if ann["image_id"] not in split_old_ids:
                continue
            new_ann = dict(ann)
            new_ann["id"] = ann_id
            new_ann["image_id"] = old_to_new_img_id[ann["image_id"]]
            new_annotations.append(new_ann)
            ann_id += 1

        out = {
            "info": info,
            "licenses": licenses,
            "images": new_images,
            "categories": categories,
            "annotations": new_annotations
        }

        save_json(out, split_dir / "annotations.json")

        print(f"✅ {split_name}: images={len(new_images)} | annots={len(new_annotations)} -> {split_dir}")

    build_split("train", train_images)
    build_split("val", val_images)


coco_json = "result.json"
images_dir = "dataset/images/train/"
output_root = "output"

split_coco_80_20(coco_json, images_dir,output_root)

✅ train: images=1600 | annots=38961 -> output\train
✅ val: images=400 | annots=9883 -> output\val


In [1]:
import json
import random
import shutil
from pathlib import Path

def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2))

def split_coco(
    coco_json_path: str,
    images_dir: str,
    output_root: str,
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    use_symlinks=False
):
    assert abs((train_ratio + val_ratio + test_ratio) - 1.0) < 1e-9, "Ratios must sum to 1.0"

    coco_json_path = Path(coco_json_path)
    images_dir = Path(images_dir)
    output_root = Path(output_root)

    coco = json.loads(coco_json_path.read_text())
    images = coco["images"]
    annotations = coco["annotations"]
    categories = coco.get("categories", [])
    info = coco.get("info", {})
    licenses = coco.get("licenses", [])

    rng = random.Random()
    imgs = images.copy()
    rng.shuffle(imgs)

    n = len(imgs)
    n_train = int(train_ratio * n)
    n_val = int(val_ratio * n)
    train_imgs = imgs[:n_train]
    val_imgs = imgs[n_train:n_train + n_val]
    test_imgs = imgs[n_train + n_val:]

    def build_split(split_name: str, split_images: list):
        split_dir = output_root / split_name
        split_img_dir = split_dir / "images"
        split_img_dir.mkdir(parents=True, exist_ok=True)

        old_to_new_img_id = {}
        new_images = []
        for new_id, img in enumerate(split_images, start=1):
            old_to_new_img_id[img["id"]] = new_id
            new_img = dict(img)
            new_img["id"] = new_id
            new_images.append(new_img)

            src = images_dir / img["file_name"]
            if not src.exists():
                raise FileNotFoundError(f"Image manquante: {src}")
            dst = split_img_dir / img["file_name"]

            if dst.exists():
                continue

            if use_symlinks:
                dst.symlink_to(src.resolve())
            else:
                shutil.copy2(src, dst)

        split_old_ids = set(old_to_new_img_id.keys())
        new_annotations = []
        ann_id = 1
        for ann in annotations:
            if ann["image_id"] not in split_old_ids:
                continue
            new_ann = dict(ann)
            new_ann["id"] = ann_id
            new_ann["image_id"] = old_to_new_img_id[ann["image_id"]]
            new_annotations.append(new_ann)
            ann_id += 1

        out = {
            "info": info,
            "licenses": licenses,
            "images": new_images,
            "categories": categories,
            "annotations": new_annotations
        }

        save_json(out, split_dir / "_annotations.coco.json")
        print(f"✅ {split_name}: images={len(new_images)} | annots={len(new_annotations)} -> {split_dir}")

    # RF-DETR préfère souvent: train / valid / test
    build_split("train", train_imgs)
    build_split("valid", val_imgs)
    build_split("test", test_imgs)


coco_json = "result.json"
images_dir = "dataset/images/train/"
output_root = "output"

split_coco(coco_json, images_dir, output_root)

✅ train: images=1600 | annots=39064 -> output\train
✅ valid: images=200 | annots=4865 -> output\valid
✅ test: images=200 | annots=4915 -> output\test


In [4]:
import json
from pathlib import Path

def fix_category_ids(split_dir):
    split_dir = Path(split_dir)
    json_path = split_dir / "_annotations.coco.json"

    data = json.loads(json_path.read_text())

    # Forcer catégorie id = 0
    data["categories"] = [{
        "id": 0,
        "name": "microbulle",
        "supercategory": "none"
    }]

    # Modifier toutes les annotations
    for ann in data["annotations"]:
        ann["category_id"] = 0

    json_path.write_text(json.dumps(data, indent=2))
    print(f"✅ corrigé: {json_path}")

root = Path(r"C:\Users\thoma\dev_projet\ecptr\test RF\output")
for split in ["train", "valid", "test"]:
    fix_category_ids(root / split)

✅ corrigé: C:\Users\thoma\dev_projet\ecptr\test RF\output\train\_annotations.coco.json
✅ corrigé: C:\Users\thoma\dev_projet\ecptr\test RF\output\valid\_annotations.coco.json
✅ corrigé: C:\Users\thoma\dev_projet\ecptr\test RF\output\test\_annotations.coco.json


In [1]:
from rfdetr import RFDETRBase

# Modèle pré-entraîné COCO
model = RFDETRBase()

model.train(
    dataset_dir="output",   # dossier racine contenant train/ et val/
    epochs=60,
    batch_size=4,
    lr=1e-4,
    output_dir="runs/microbulle_rfdetr"
)

KeyboardInterrupt: 